# Adversarial validation — train_soundscapes vs test_soundscapes

Tests how distinguishable the training audio is from the LB test set, using
Perch embeddings as features. High adv-AUC (>0.85) means there's a real
domain gap and pseudo-labeling on `train_soundscapes/` is structurally
limited. Low (<0.6) means distributions overlap and the CNN's LB regression
came from something else.

Outputs:
- Headline `adv_auc` (5-fold mean ± std)
- `train_test_likeness.npz` with per-row test-likeness scores for every
  train_soundscape row → can be used as a pseudo-label filter in round 2.


In [ ]:
# ── Cell 0: Install (onnxruntime + xgboost — usually present on Kaggle) ───
import subprocess, sys
try:
    import onnxruntime as ort
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "onnxruntime"])
    import onnxruntime as ort
try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
    import xgboost as xgb
print(f"onnxruntime {ort.__version__} | xgboost {xgb.__version__}")


In [ ]:
# ── Cell 1: Imports & paths ──────────────────────────────────────────────
import os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

# === Paths — edit if your dataset mounts are named differently ==========
PERCH_ONNX = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/perch_v2.onnx")
TRAIN_CACHE_META = Path("/kaggle/input/datasets/majkel1337/perch-cache/full_perch_meta.parquet")
TRAIN_CACHE_NPZ  = Path("/kaggle/input/datasets/majkel1337/perch-cache/full_perch_arrays.npz")
TEST_DIR = Path("/kaggle/input/birdclef-2026/test_soundscapes")
SAMPLE_SUB = Path("/kaggle/input/birdclef-2026/sample_submission.csv")
SOUNDSCAPE_LABELS = Path("/kaggle/input/birdclef-2026/train_soundscapes_labels.csv")
WORK = Path("/kaggle/working")

SR = 32_000
N_WINDOWS = 12
WINDOW_SAMPLES = SR * 5
FILE_SAMPLES = SR * 60

# Resolve fallbacks if the default Perch path is missing
if not PERCH_ONNX.exists():
    for alt in [
        "/kaggle/input/perch-onnx/perch_v2.onnx",
        "/kaggle/working/perch_v2.onnx",
    ]:
        if Path(alt).exists():
            PERCH_ONNX = Path(alt); break
assert PERCH_ONNX.exists(), f"Perch ONNX not found. Tried {PERCH_ONNX}"
print(f"perch onnx: {PERCH_ONNX}")
print(f"train cache: {TRAIN_CACHE_META.exists()}, {TRAIN_CACHE_NPZ.exists()}")
print(f"test dir: {TEST_DIR.exists()}, files: {len(list(TEST_DIR.glob('*.ogg')))}")


In [ ]:
# ── Cell 2: Load Perch ONNX ──────────────────────────────────────────────
import onnxruntime as ort

so = ort.SessionOptions()
so.intra_op_num_threads = 4
so.inter_op_num_threads = 1
sess = ort.InferenceSession(str(PERCH_ONNX), sess_options=so,
                            providers=["CPUExecutionProvider"])
INPUT_NAME = sess.get_inputs()[0].name
OUTPUT_MAP = {o.name: i for i, o in enumerate(sess.get_outputs())}
EMB_IDX = OUTPUT_MAP["embedding"]
print(f"Perch loaded. input='{INPUT_NAME}', emb_idx={EMB_IDX}")


In [ ]:
# ── Cell 3: Load TRAIN embeddings from the existing Perch cache ──────────
# We use ALL train_soundscape rows (labeled + unlabeled) so the classifier
# sees the full training distribution.

import pandas as pd, numpy as np
meta_tr = pd.read_parquet(TRAIN_CACHE_META)
arr = np.load(TRAIN_CACHE_NPZ)
emb_train = arr["emb_full"].astype(np.float32)   # (N_train, 1536)
print(f"train embeddings: shape={emb_train.shape}, files={meta_tr['filename'].nunique()}")
print(f"  labeled rows: {meta_tr['is_labeled'].sum() if 'is_labeled' in meta_tr.columns else 'n/a'}")


In [ ]:
# ── Cell 4: Run Perch on TEST soundscapes (compute embeddings inline) ───
# Each test file → 12 × 5s windows → 12 × 1536 embeddings.

def read_60s(path):
    y, _sr = sf.read(str(path), dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:
        y = y[:FILE_SAMPLES]
    return y.astype(np.float32, copy=False)


test_paths = sorted(TEST_DIR.glob("*.ogg"))
print(f"{len(test_paths)} test files found in {TEST_DIR}")

if len(test_paths) == 0:
    raise SystemExit(
        f"No test files at {TEST_DIR}. On the public sample-test branch this "
        f"is expected to be 0-1 files; on the actual scoring run it will be "
        f"the full hidden test set. Run this notebook in scoring mode to get "
        f"a meaningful adversarial AUC against the real LB distribution."
    )

emb_test_list = []
test_row_filename = []
test_row_window = []
t0 = time.time()
for p in tqdm(test_paths, desc="Perch on test"):
    y = read_60s(p)
    x = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
    outs = sess.run(None, {INPUT_NAME: x})
    emb_test_list.append(outs[EMB_IDX].astype(np.float32))
    for w in range(N_WINDOWS):
        test_row_filename.append(p.name)
        test_row_window.append(w)
emb_test = np.vstack(emb_test_list)
test_meta = pd.DataFrame({"filename": test_row_filename, "window": test_row_window})
print(f"test embeddings: shape={emb_test.shape}  ({time.time()-t0:.1f}s)")


In [ ]:
# ── Cell 5: Build adversarial dataset ────────────────────────────────────
# X = stacked features, y = 0 (train), 1 (test). Use ALL test rows + a
# matched-size random sample of train rows to keep the classes balanced
# (XGBoost handles imbalance fine but balanced is easier to interpret).

rng = np.random.default_rng(42)
n_test = emb_test.shape[0]
# Use ALL train rows — we want the classifier's per-row scores on every
# train row downstream. Imbalance is handled via scale_pos_weight.
emb_tr_all = emb_train
print(f"X_train rows: {len(emb_tr_all):,}, X_test rows: {len(emb_test):,}")
print(f"  imbalance ratio (train:test): {len(emb_tr_all) / max(1, len(emb_test)):.1f}:1")

X = np.vstack([emb_tr_all, emb_test]).astype(np.float32)
y_lbl = np.zeros(X.shape[0], dtype=np.int64)
y_lbl[len(emb_tr_all):] = 1
print(f"X: {X.shape}  positives (test): {y_lbl.sum()}")


In [ ]:
# ── Cell 6: 5-fold XGBoost — adversarial AUC ─────────────────────────────
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

aucs = []
oof_scores = np.zeros(len(X), dtype=np.float32)

scale_pos_weight = (y_lbl == 0).sum() / max(1, (y_lbl == 1).sum())
xgb_params = dict(
    n_estimators=400, max_depth=4, learning_rate=0.06,
    subsample=0.8, colsample_bytree=0.7,
    scale_pos_weight=float(scale_pos_weight),
    tree_method="hist", random_state=42, eval_metric="auc",
)

skf = StratifiedKFold(5, shuffle=True, random_state=42)
for fold, (tr, va) in enumerate(skf.split(X, y_lbl)):
    clf = xgb.XGBClassifier(**xgb_params)
    clf.fit(X[tr], y_lbl[tr], eval_set=[(X[va], y_lbl[va])], verbose=False)
    p = clf.predict_proba(X[va])[:, 1]
    oof_scores[va] = p
    auc = roc_auc_score(y_lbl[va], p)
    print(f"fold {fold}  adv-AUC = {auc:.4f}  ({(y_lbl[va]==1).sum()} test in val)")
    aucs.append(auc)

adv_auc_mean = float(np.mean(aucs))
adv_auc_std  = float(np.std(aucs))
print()
print(f"=== Adversarial AUC (train_soundscapes vs test_soundscapes) ===")
print(f"  mean ± std = {adv_auc_mean:.4f} ± {adv_auc_std:.4f}")


In [ ]:
# ── Cell 7: Interpret the adversarial AUC ────────────────────────────────
print(f"adv-AUC = {adv_auc_mean:.3f} ± {adv_auc_std:.3f}")
print()
if adv_auc_mean < 0.60:
    print("[A] adv-AUC < 0.60 — distributions overlap. Pseudo failed for OTHER reasons")
    print("    (label quality, calibration shift, teacher overfit). Investigate those")
    print("    rather than filtering the unlabeled pool.")
elif adv_auc_mean < 0.85:
    print("[B] adv-AUC ∈ [0.60, 0.85] — moderate domain gap. Heterogeneous unlabeled pool.")
    print("    Filter pseudo to top X% test-like rows. Expected LB lift if it works:")
    print("    +0.005 to +0.015. The cell below dumps per-row scores you can use.")
else:
    print("[C] adv-AUC > 0.85 — strong domain gap. Filtering unlikely to recover much.")
    print("    Pivot to architectural diversification (SED) or different data sources")
    print("    (Pantanal-region XC). The unlabeled pool is structurally not LB-similar.")


In [ ]:
# ── Cell 8: Score every TRAIN row for "test-likeness" + dump npz ─────────
# Train one final classifier on ALL data, get per-row scores. These are
# the filter signal for round-2 pseudo-labeling.

clf_full = xgb.XGBClassifier(**xgb_params)
clf_full.fit(X, y_lbl, verbose=False)
all_scores = clf_full.predict_proba(X)[:, 1]   # higher = more test-like

train_test_likeness = all_scores[:len(emb_tr_all)]   # per-row score for train rows
test_self_scores    = all_scores[len(emb_tr_all):]   # sanity: test rows should score high

print(f"train rows: median test-likeness = {np.median(train_test_likeness):.4f}")
print(f"test  rows: median test-likeness = {np.median(test_self_scores):.4f}")

# Quantiles of train test-likeness
q = np.percentile(train_test_likeness, [10, 25, 50, 75, 90, 95, 99])
print(f"train test-likeness percentiles: 10%={q[0]:.3f} 25%={q[1]:.3f} 50%={q[2]:.3f} "
      f"75%={q[3]:.3f} 90%={q[4]:.3f} 95%={q[5]:.3f} 99%={q[6]:.3f}")

# How many train rows clear common filter thresholds
for thr in [0.5, 0.7, 0.8, 0.9, 0.95]:
    n = int((train_test_likeness >= thr).sum())
    print(f"train rows with test-likeness >= {thr}: {n:,}  ({n/len(train_test_likeness)*100:.1f}%)")

out_npz = WORK / "train_test_likeness.npz"
np.savez_compressed(
    out_npz,
    train_test_likeness=train_test_likeness.astype(np.float32),
    train_filenames=meta_tr["filename"].astype(str).to_numpy(),
    test_self_scores=test_self_scores.astype(np.float32),
    test_filenames=test_meta["filename"].astype(str).to_numpy(),
    test_windows=test_meta["window"].to_numpy(),
    adv_auc_mean=np.float32(adv_auc_mean),
    adv_auc_std=np.float32(adv_auc_std),
)
print(f"\nwrote {out_npz}  (use as pseudo-label filter in round 2)")


In [ ]:
# ── Cell 9: (Optional) Visualize the distribution of test-likeness ──────
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))
    ax.hist(train_test_likeness, bins=80, alpha=0.6, label="train rows", density=True)
    ax.hist(test_self_scores,    bins=20, alpha=0.6, label="test rows",  density=True)
    ax.axvline(0.5, color="red", linestyle="--", alpha=0.5, label="threshold = 0.5")
    ax.set_xlabel("test-likeness score (P(row is from test_soundscapes))")
    ax.set_ylabel("density")
    ax.set_title(f"adv-AUC = {adv_auc_mean:.3f} ± {adv_auc_std:.3f}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(WORK / "test_likeness_hist.png", dpi=120)
    plt.show()
except ImportError:
    print("matplotlib not available, skipping plot")
